# CM2604 Machine Learning – Part 01: Telco Customer Churn
Binary classification (churn = Yes/No) with **KNN**, **Logistic Regression** and **Random Forest**.

**Dataset:** Telco Customer Churn (Kaggle, IBM sample data) – save the CSV as `data/WA_Fn-UseC_-Telco-Customer-Churn.csv`.

**Experimental settings compared**
- **Setting A – Baseline:** original (imbalanced) training data.
- **Setting B – Imbalance handling:** SMOTE oversampling applied *inside* each CV fold (training folds only, so no leakage).

**Workflow:** 1) Load & audit → 2) EDA (Task 1) → 3) Preprocessing → 4) Tuning & training (Task 2) → 5) Evaluation → 6) Fairness check & ethics (Task 3)

## 0. Setup

In [ ]:
import warnings, os, random
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             ConfusionMatrixDisplay, RocCurveDisplay, classification_report)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

SEED = 42
np.random.seed(SEED); random.seed(SEED)
sns.set_theme(style="whitegrid")
os.makedirs("figures", exist_ok=True); os.makedirs("results", exist_ok=True)

DATA_PATH = "data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
FAST = False   # True = smaller grids for a quick test run

## 1. Load and audit the data

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate customerIDs:", df["customerID"].duplicated().sum())

`TotalCharges` is stored as text in the raw file because a few rows contain blank strings (customers with `tenure = 0`, i.e. brand-new customers who have not been billed yet). We convert it to numeric and inspect what became missing.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])
print("\nRows with missing TotalCharges (tenure values):")
print(df.loc[df["TotalCharges"].isna(), ["customerID", "tenure", "MonthlyCharges", "Churn"]])

In [ ]:
# Missing TotalCharges only occurs when tenure == 0 -> no charges billed yet, so 0 is a justified fill value.
assert (df.loc[df["TotalCharges"].isna(), "tenure"] == 0).all()
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# Consistent categories: 'No internet service' / 'No phone service' are equivalent to 'No' for the add-on services
service_cols = ["MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
                "TechSupport", "StreamingTV", "StreamingMovies"]
for c in service_cols:
    df[c] = df[c].replace({"No internet service": "No", "No phone service": "No"})

df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})
df.describe(include="all").T

## 2. Task 1 – Exploratory Data Analysis

### 2.1 Target distribution

In [ ]:
counts = df["Churn"].value_counts()
print(counts, "\n", (counts / len(df) * 100).round(2))
fig, ax = plt.subplots(figsize=(4.5, 4))
sns.countplot(data=df, x="Churn", palette=["#4C72B0", "#C44E52"], ax=ax)
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x()+p.get_width()/2, p.get_height()), ha="center", va="bottom")
ax.set_title("Class distribution"); plt.tight_layout()
plt.savefig("figures/01_class_distribution.png", dpi=150); plt.show()

**Insight:** the classes are imbalanced (roughly 73% stay / 27% churn). A model that always predicts 'No' would score ~73% accuracy, so accuracy alone is misleading. We therefore evaluate with precision, recall, F1, ROC-AUC and PR-AUC, and compare against an imbalance-handling setting.

### 2.2 Numeric features vs churn

In [ ]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, c in enumerate(num_cols):
    sns.histplot(data=df, x=c, hue="Churn", kde=True, bins=30, ax=axes[0, i], palette=["#4C72B0", "#C44E52"])
    axes[0, i].set_title(f"{c} distribution by churn")
    sns.boxplot(data=df, x="Churn", y=c, ax=axes[1, i], palette=["#4C72B0", "#C44E52"])
    axes[1, i].set_title(f"{c} by churn")
plt.tight_layout(); plt.savefig("figures/02_numeric_vs_churn.png", dpi=150); plt.show()
print(df.groupby("Churn")[num_cols].agg(["mean", "median"]).round(2))

**Insights to note in the report:** churners have much shorter tenure (they leave early) and higher monthly charges; `TotalCharges` is strongly tied to tenure (multicollinearity risk for Logistic Regression).

### 2.3 Categorical features vs churn rate

In [ ]:
cat_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c]) and c not in ["customerID", "Churn"]]
df["_churn"] = (df["Churn"] == "Yes").astype(int)
n = len(cat_cols); ncols = 4; nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(20, 4.2 * nrows))
for ax, c in zip(axes.ravel(), cat_cols):
    rate = df.groupby(c)["_churn"].mean().sort_values(ascending=False) * 100
    sns.barplot(x=rate.index, y=rate.values, ax=ax, color="#C44E52")
    ax.set_title(f"Churn rate (%) by {c}"); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=25)
for ax in axes.ravel()[n:]: ax.axis("off")
plt.tight_layout(); plt.savefig("figures/03_categorical_vs_churn.png", dpi=150); plt.show()
df = df.drop(columns="_churn")

**Insights to note:** month-to-month contracts, fibre-optic internet, electronic-check payment, no tech support / online security, and senior citizens show the highest churn rates. Gender and phone service show little difference.

### 2.4 Correlation and multicollinearity

In [ ]:
tmp = df.copy()
tmp["Churn"] = (tmp["Churn"] == "Yes").astype(int)
corr = tmp[["tenure", "MonthlyCharges", "TotalCharges", "Churn"]].corr()
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation (numeric features + churn)"); plt.tight_layout()
plt.savefig("figures/04_correlation.png", dpi=150); plt.show()

## 3. Data preprocessing and feature engineering

Decisions (justify each in the report):
- Drop `customerID` (identifier, no predictive meaning, would cause overfitting).
- Encode target: Yes → 1, No → 0.
- **Feature engineering:** `num_services` (count of add-on services), `avg_monthly_spend` (= TotalCharges / max(tenure,1)).
- **Stratified 80/20 train/test split** to preserve the churn ratio; test set is untouched until the final evaluation.
- One-hot encode categoricals; **scale numerics for KNN and Logistic Regression** (distance/gradient-based, scale-sensitive). **Random Forest uses unscaled numerics** (tree splits are scale-invariant).
- All preprocessing lives inside sklearn Pipelines so it is fitted on training folds only (no data leakage).

In [ ]:
data = df.drop(columns="customerID").copy()
data["Churn"] = (data["Churn"] == "Yes").astype(int)

addon_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
data["num_services"] = (data[addon_cols] == "Yes").sum(axis=1)
data["avg_monthly_spend"] = data["TotalCharges"] / data["tenure"].clip(lower=1)

X = data.drop(columns="Churn"); y = data["Churn"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Churn rate train/test:", y_train.mean().round(4), y_test.mean().round(4))

num_features = ["tenure", "MonthlyCharges", "TotalCharges", "num_services", "avg_monthly_spend"]
cat_features = [c for c in X.columns if c not in num_features]
print("Numeric:", num_features); print("Categorical:", cat_features)

In [ ]:
def make_preprocessor(scale: bool):
    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale: num_steps.append(("scaler", StandardScaler()))
    return ColumnTransformer([
        ("num", ImbPipeline(num_steps), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), cat_features),
    ])

## 4. Task 2 – Model implementation and hyperparameter tuning
Tuning: `GridSearchCV`, 5-fold **stratified** CV on the training set, optimising **F1 of the churn class** (balances precision and recall on the minority class).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

models = {
    "KNN": dict(
        est=KNeighborsClassifier(), scale=True,
        grid={"clf__n_neighbors": [5, 11, 21, 31] if FAST else [3, 5, 7, 11, 15, 21, 31],
              "clf__weights": ["uniform", "distance"],
              "clf__p": [1, 2]}),
    "Logistic Regression": dict(
        est=LogisticRegression(max_iter=2000, solver="liblinear", random_state=SEED), scale=True,
        grid={"clf__C": [0.01, 0.1, 1, 10] if FAST else [0.001, 0.01, 0.1, 1, 10, 100],
              "clf__penalty": ["l1", "l2"]}),
    "Random Forest": dict(
        est=RandomForestClassifier(random_state=SEED, n_jobs=-1), scale=False,
        grid={"clf__n_estimators": [200] if FAST else [200, 400],
              "clf__max_depth": [8, None] if FAST else [6, 10, 14, None],
              "clf__min_samples_leaf": [1, 5] if FAST else [1, 3, 5, 10],
              "clf__max_features": ["sqrt"]}),
}

def build_pipeline(name, use_smote):
    m = models[name]
    steps = [("prep", make_preprocessor(m["scale"]))]
    if use_smote: steps.append(("smote", SMOTE(random_state=SEED)))
    steps.append(("clf", m["est"]))
    return ImbPipeline(steps)

settings = {"A: Baseline": False, "B: SMOTE": True}
fitted, tuning_rows = {}, []
for sname, use_smote in settings.items():
    for name in models:
        gs = GridSearchCV(build_pipeline(name, use_smote), models[name]["grid"],
                          scoring="f1", cv=cv, n_jobs=-1, refit=True)
        gs.fit(X_train, y_train)
        fitted[(sname, name)] = gs
        tuning_rows.append({"Setting": sname, "Model": name, "Best CV F1": round(gs.best_score_, 4),
                            "Best params": {k.replace("clf__", ""): v for k, v in gs.best_params_.items()}})
        print(f"{sname:12s} | {name:20s} | CV F1 = {gs.best_score_:.4f} | {gs.best_params_}")
tuning_df = pd.DataFrame(tuning_rows)
tuning_df.to_csv("results/tuning_results.csv", index=False)
tuning_df

## 5. Model evaluation on the held-out test set
Metrics: accuracy, precision, recall, F1 (churn = positive class), ROC-AUC, PR-AUC. **Recall matters most in a churn scenario** (missing a churner costs more than a wasted retention offer), but F1 and PR-AUC keep precision honest.

In [ ]:
rows, preds = [], {}
for (sname, name), gs in fitted.items():
    yp = gs.predict(X_test); ys = gs.predict_proba(X_test)[:, 1]
    preds[(sname, name)] = (yp, ys)
    rows.append({"Setting": sname, "Model": name,
                 "Accuracy": accuracy_score(y_test, yp), "Precision": precision_score(y_test, yp),
                 "Recall": recall_score(y_test, yp), "F1": f1_score(y_test, yp),
                 "ROC-AUC": roc_auc_score(y_test, ys), "PR-AUC": average_precision_score(y_test, ys)})
results = pd.DataFrame(rows).round(4)
results.to_csv("results/test_results.csv", index=False)
results

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
melt = results.melt(id_vars=["Setting", "Model"], value_vars=["Precision", "Recall", "F1", "ROC-AUC"])
sns.barplot(data=melt, x="variable", y="value", hue=melt["Model"] + " | " + melt["Setting"], ax=ax)
ax.set_ylim(0, 1); ax.set_xlabel(""); ax.set_ylabel("Score"); ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
ax.set_title("Test-set comparison"); plt.tight_layout()
plt.savefig("figures/05_metric_comparison.png", dpi=150); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for r, sname in enumerate(settings):
    for c, name in enumerate(models):
        yp, _ = preds[(sname, name)]
        ConfusionMatrixDisplay(confusion_matrix(y_test, yp), display_labels=["Stay", "Churn"]).plot(ax=axes[r, c], colorbar=False, cmap="Blues")
        axes[r, c].set_title(f"{name}\n{sname}")
plt.tight_layout(); plt.savefig("figures/06_confusion_matrices.png", dpi=150); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, sname in zip(axes, settings):
    for name in models:
        RocCurveDisplay.from_predictions(y_test, preds[(sname, name)][1], name=name, ax=ax)
    ax.plot([0, 1], [0, 1], "k--", alpha=.5); ax.set_title(f"ROC curves – {sname}")
plt.tight_layout(); plt.savefig("figures/07_roc_curves.png", dpi=150); plt.show()

In [ ]:
best_key = results.sort_values("F1", ascending=False).iloc[0][["Setting", "Model"]].tolist()
best_key = tuple(best_key); print("Best by test F1:", best_key)
print(classification_report(y_test, preds[best_key][0], target_names=["Stay", "Churn"]))

### 5.1 Interpretability – what drives churn?

In [ ]:
def feature_names(gs):
    return gs.best_estimator_.named_steps["prep"].get_feature_names_out()

rf = fitted[("A: Baseline", "Random Forest")]
imp = pd.Series(rf.best_estimator_.named_steps["clf"].feature_importances_, index=feature_names(rf)).sort_values(ascending=False).head(12)
lr = fitted[("A: Baseline", "Logistic Regression")]
coef = pd.Series(lr.best_estimator_.named_steps["clf"].coef_[0], index=feature_names(lr))
coef = coef.reindex(coef.abs().sort_values(ascending=False).head(12).index)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
imp[::-1].plot.barh(ax=axes[0], color="#55A868"); axes[0].set_title("Random Forest – top feature importances")
coef[::-1].plot.barh(ax=axes[1], color=["#C44E52" if v > 0 else "#4C72B0" for v in coef[::-1]])
axes[1].set_title("Logistic Regression – top coefficients (red = more churn)")
plt.tight_layout(); plt.savefig("figures/08_feature_importance.png", dpi=150); plt.show()

## 6. Task 3 – AI ethics
### 6.1 Fairness check across sensitive groups
Sensitive attributes were **not removed silently**: we keep them out of decisions by auditing outcomes per group instead. We compare recall, false-positive rate and selection rate for `gender` and `SeniorCitizen` on the test set using the best model.

In [ ]:
def group_report(attr):
    yp = preds[best_key][0]; out = []
    for g in X_test[attr].unique():
        m = (X_test[attr] == g).values
        tn, fp, fn, tp = confusion_matrix(y_test[m], yp[m], labels=[0, 1]).ravel()
        out.append({attr: g, "n": m.sum(), "Actual churn rate": y_test[m].mean().round(3),
                    "Selection rate": yp[m].mean().round(3),
                    "Recall (TPR)": round(tp / (tp + fn), 3) if tp + fn else np.nan,
                    "False-positive rate": round(fp / (fp + tn), 3) if fp + tn else np.nan})
    return pd.DataFrame(out)
for a in ["gender", "SeniorCitizen"]:
    display(group_report(a))

Interpretation guide for the report: large gaps in recall / FPR / selection rate between groups indicate disparate treatment risk. Some gap in `SeniorCitizen` may reflect real differences in base churn rate, but it should be monitored and discussed, not ignored.

### 6.2 Development-stage strategies followed (edit to match what you actually did)
- **Data minimisation and privacy:** dropped `customerID`; no names, addresses or contact details used; data is anonymised sample data.
- **Bias awareness:** audited class imbalance and per-group performance (gender, senior citizen); did not use group membership to set different thresholds.
- **Fair evaluation:** stratified split, leakage-free pipelines, test set touched only once; metrics chosen for the minority class rather than accuracy.
- **Transparency:** interpretable baseline (Logistic Regression) plus feature importance for Random Forest; all code and seeds version-controlled in Git for reproducibility.
- **Responsible use:** predictions are decision support for retention offers, not for denying service or changing prices for individuals.

### 6.3 Post-deployment strategies
- **Monitoring for drift:** track input distributions (e.g. PSI) and live precision/recall against real churn outcomes; alert on degradation.
- **Scheduled retraining and versioning** of models, data and thresholds, with rollback.
- **Ongoing fairness audits** per demographic group, with agreed tolerance limits.
- **Human oversight and recourse:** retention staff review high-impact actions; customers can contest and opt out of profiling.
- **Privacy and governance:** access control, retention limits and legal compliance (e.g. PDPA in Sri Lanka, GDPR if EU customers are involved); documentation via a model card.
- **Feedback loop caution:** retention offers change future behaviour, so evaluate with holdout/control groups to avoid self-fulfilling labels.

## 7. Limitations and future enhancements (starting points for the report)
**Limitations:** single dataset from one provider, no time dimension (cannot test temporal drift), SMOTE on one-hot features creates synthetic in-between categorical values, F1-optimised default 0.5 threshold, limited grid search, no cost-sensitive analysis, only two sensitive attributes audited.

**Future work:** cost-based threshold selection, gradient boosting / calibrated probabilities, SMOTE-NC or class-weighting comparison, randomised/Bayesian search, SHAP explanations, time-based validation, fairness-constrained training.